# Module 8 Assignment: Give Your Persona Agent Memory With Sessions

Lab 7 added a `SQLiteSession` to Study Buddy: first in memory, to fix two `Runner.run` calls that
did not remember each other, then file backed, so the conversation survives a kernel restart. The
Gradio `chat()` function got simpler in the process, no more rebuilding `history` by hand.

This assignment makes you do the same thing to your **own** Persona Agent, the one you rebuilt on
the OpenAI Agents SDK in the Module 7 assignment. Nothing about your agent's persona, notes, or
three tools needs to change here: you are giving it memory, not redesigning it.

**What you'll build:**
1. Your Module 7 assignment's Agent, Azure configuration, and three tools, carried over unchanged
2. A quick demonstration that two separate `Runner.run` calls do not remember each other
3. A file backed `SQLiteSession` that fixes it
4. A simplified Gradio `chat()` function that uses `session=session` instead of rebuilding `history`

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu`, same as every lab and assignment so
far.

If you get stuck on any step, look at how Lab 7 made the same kind of change to Study Buddy, and do
the analogous thing to your own agent.

## 0. Install the packages (run once)

Same as Lab 7: no new package beyond what your Module 7 assignment already installed.
`SQLiteSession` ships with `openai-agents` itself.

In [1]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio

Resolved 75 packages in 948ms                                        
Prepared 70 packages in 262ms                                                openai               ------------------------------ 1.79 MiB/2.15 MiB           openai               ------------------------------ 144.00 KiB/2.15 MiB         openai               ------------------------------     0 B/2.15 MiB            
Installed 70 packages in 309ms                              
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anyio==4.15.1
 + attrs==26.1.0
 + brotli==1.2.0
 + certifi==2026.7.22
 + cffi==2.1.1
 + charset-normalizer==3.5.2
 + click==8.5.0
 + cryptography==50.0.2
 + fastapi==0.143.3
 + filelock==4.2.0
 + fsspec==2026.9.0
 + gradio==6.30.0
 + gradio-client==2.7.2
 + griffelib==2.3.2
 + groovy==0.1.2
 + h11==0.16.0
 + hf-gradio==0.4.1
 + hf-xet==1.7.0
 + httpcore==1.0.9
 + httpcore2==2.13.1
 + httpx==0.28.1
 + httpx2==2.13.1
 + huggingface-hub==2.2.0
 + idna==3.20
 + jinja2==3.1.6
 + jiter==0.17.0
 + js

## 1. Imports

**TODO:** Copy the imports from your Module 7 assignment, plus one new name: `SQLiteSession`,
imported from `agents` alongside `Agent` and `Runner`, the same way Lab 7, Step 1 added it.

In [2]:
# TODO: copy your imports from the Module 7 assignment, plus SQLiteSession from agents
from dotenv import load_dotenv
import os
import asyncio
import html
from typing import Literal, Optional

import requests
from openai import AsyncAzureOpenAI
from agents import (
    Agent,
    Runner,
    function_tool,
    OpenAIChatCompletionsModel,
    set_tracing_disabled,
    SQLiteSession,
)
from pypdf import PdfReader
import gradio as gr

/Users/kobe/Senior/IT3025C - Introduction to Agentic AI/Week 8/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load your Azure OpenAI credentials from `.env`

Identical to the Module 7 assignment. Sessions do not change which provider you talk to.

In [3]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

Endpoint loaded: True
API key loaded: True


## 3. Configure the OpenAI Agents SDK for Azure OpenAI

**TODO:** Copy this setup from your Module 7 assignment, unchanged: `set_tracing_disabled(True)`,
your `AsyncAzureOpenAI` client, and `OpenAIChatCompletionsModel`. A session is a feature of
`Runner.run`, not of the model or the client, so nothing here needs to change.

In [4]:
# TODO: set_tracing_disabled(True), then build azure_client and azure_model, exactly like your Module 7 assignment
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
)

azure_model = OpenAIChatCompletionsModel(
    model=DEPLOYMENT,
    openai_client=azure_client,
)

print("Azure model configured for the OpenAI Agents SDK.")

Azure model configured for the OpenAI Agents SDK.


## 4. Your persona, notes, and three tools

**TODO:** Bring over your persona, your notes, and all three of your `@function_tool` decorated
tools from the Module 7 assignment, unchanged. None of them need to change for this assignment: a
session only affects how conversation history is stored and replayed, not what a tool does.

In [5]:
# TODO: persona = "..." (a parenthesized multi-line string), from your Module 7 assignment
# TODO: if your persona uses notes, reader = PdfReader("notes/notes.pdf"); notes = ""; loop over
#       reader.pages and build up notes the same way Lab 6 did, if applicable
NOTES_PATH = 'notes/notes.pdf'

def load_notes(path):
    try:
        reader = PdfReader(path)
        text = "\n".join(page.extract_text() or "" for page in reader.pages)
        return text.strip()
    except FileNotFoundError:
        return "(notes.pdf not found yet: add it to this folder to enable real study content.)"

study_notes = load_notes(NOTES_PATH)

persona = ("Coliver Assistant, a collaborative assistant that provides constructive feedback, supports learning, and gives helpful hints when users answer incorrectly or are unsure how to respond.")
print(study_notes[:400])

Data Technology Study Notes
Topic: Core Concepts in Modern Data Systems
Databases and Data Models

Relational databases organize data into tables made of rows and columns. SQL is commonly used to
query and modify this data.

Primary keys uniquely identify rows, while foreign keys connect related tables.

NoSQL databases are useful when data is semi-structured, rapidly changing, or does not fit 


In [6]:
# TODO: your first tool, decorated with @function_tool, from your Module 7 assignment
@function_tool
def bookmark_topic_tool(topic: str) -> str:
    # TODO: implement your tool. For example, print something and/or write it to a file.
    # Return a short string describing what happened.
    with open("review_topics.txt", "a") as file:
        file.write(f"Topic: {topic}\n")
    return f"Bookmarked topic: {topic}"


In [7]:
# TODO: your second tool, decorated with @function_tool, from your Module 7 assignment
@function_tool
def simplify_topic_tool(topic: str) -> str:
    topic = " ".join(str(v) for v in value).strip()

    if not topic:
        return "No topic was provided."

    url = "https://en.wikipedia.org/w/rest.php/v1/search/page"

    params = {
        "q": topic,
        "limit": 1
    }

    headers = {
        "User-Agent": "StudyAssistant"
    }

    try:
        response = requests.get(url, params=params,headers=headers, timeout=5)
        response.raise_for_status()

        data = response.json()
        pages = data.get("pages", [])

        if not pages:
            return f"No information was found for '{topic}'."

        page = pages[0]

        title = page.get("title", topic)
        description = page.get("description", "")
        excerpt = page.get("excerpt", "")

        # Remove simple HTML tags from excerpt
        excerpt = (
            excerpt.replace("<span class=\"searchmatch\">", "")
            .replace("</span>", "")
        )

        return f"{title}: {description}. {excerpt}"

    except requests.exceptions.RequestException:
        return "The Wikipedia service is currently unavailable."

In [8]:
# TODO: your third tool, decorated with @function_tool, from your Module 7 assignment
@function_tool
def follow_up_question_tool(topic: str, level: str = "medium") -> str:
    topic = str(topic).strip()

    if not topic:
        return "No topic provided."

    return (
        f"Ask the student one follow-up question about '{topic}' "
        f"at {level} difficulty. Do not give the answer yet."
    )

## 5. Build your Agent

**TODO:** Copy this from your Module 7 assignment, unchanged: `name`, `instructions`, your `tools`
list, and `model=azure_model`.

In [10]:
# TODO: instructions = f"""..."""  (your # Your role / # Context / # Rules structured template,
#       using {persona} and {notes}, from your Module 7 assignment)
# TODO: your_agent = Agent(name=..., instructions=instructions, tools=[...], model=azure_model)
instructions = f"""
# Your role
You are a collaborative study assistant.

Persona:
{persona}

# Context
Study notes:
{study_notes}

# Rules

When identifying a topic for a tool call, always use a short and accurate
topic name, such as "SQL vs NoSQL", "DNS", or "Python Lists".

If the student answers incorrectly, says they do not know,
or clearly struggles with a topic, call bookmark_topic_tool
to save that topic for future review.

Only bookmark the topic once per struggle.

If the student is confused, asks for a simpler explanation,
or continues to struggle, call simplify_topic_tool with the topic.

If the student answers correctly and additional practice would be useful,
call follow_up_question_tool with the same topic.

Use:
- "easy" if the student is struggling
- "medium" for normal practice
- "hard" if the student is doing well

Do not give the answer to a follow-up question immediately.
Wait for the student to answer first.
"""

study_assistant = Agent(
    name="Coliver Assistant",
    instructions=instructions,
    tools=[bookmark_topic_tool, simplify_topic_tool, follow_up_question_tool],
    model=azure_model,
)

print("Agent built:", study_assistant.name)


Agent built: Coliver Assistant


## 6. Confirm the problem: two calls that do not know each other

**TODO:** Run your agent twice in a row with `await Runner.run(your_agent, "...")`, the same way
Lab 7, Step 7 did with Study Buddy. Pick a first message that gives the agent some information (for
example, ask it a question whose answer it needs to remember), then a second message that only
makes sense if the agent remembers the first (for example, "what did I just ask you?" or "say that
again"). Confirm the second reply shows the agent has no idea what you mean.

In [13]:
# TODO: result = await Runner.run(your_agent, "...")
# TODO: print(result.final_output)
result = await Runner.run(study_assistant, "Hi! Can you quiz me with one question from my notes?")
print(result.final_output)

Sure — here’s one:

What is the difference between OLTP systems and OLAP systems?


In [19]:
# TODO: result = await Runner.run(your_agent, "...a follow up that only makes sense with memory...")
# TODO: print(result.final_output)
result = await Runner.run(study_assistant, "Actually, I don't know the answer to that one.")
print(result.final_output)

No problem — this topic is worth reviewing.

Quick version:
- **OLTP** systems handle everyday operations like inserts, updates, and transactions.
- **OLAP** systems handle analysis, reporting, and big queries over lots of historical data.
- A **data warehouse** is a system that stores integrated historical data and is usually designed for **OLAP-style** work.

A simple way to remember it:
- **OLTP = running the business**
- **OLAP/data warehouse = analyzing the business**

Want to try a very short practice question on this?


## 7. Fix it with a file backed `SQLiteSession`

**TODO:** Create a `SQLiteSession`, the same way Lab 7, Step 9 did: give it a `session_id` string,
and a `db_path` so the conversation is written to a file and survives a kernel restart. Then repeat
the same two messages from Step 6, this time passing `session=your_session` to both calls, and
confirm the second reply now correctly remembers the first.

In [15]:
# TODO: your_session = SQLiteSession("your_session_id", "your_conversation_history.db")
study_session = SQLiteSession("study_session","study_session.db")

result = await Runner.run(
    study_assistant,
    "Hi! Can you quiz me with one question from my notes?",
    session=study_session,
)
print(result.final_output)

Sure — here’s one:

What is the difference between OLTP systems and OLAP systems?


In [16]:
# TODO: result = await Runner.run(your_agent, "...", session=your_session)
# TODO: print(result.final_output)
result = await Runner.run(study_assistant, "I don't know the answer for this one?", session=study_session)
print(result.final_output)

No problem — this is a good one to review.

OLTP systems are for day-to-day operations, where data is frequently inserted or updated.

OLAP systems are for analysis, where people run complex queries over large amounts of historical data.

A simple way to remember it:
- OLTP = running the business
- OLAP = analyzing the business

Want me to give you a simpler follow-up question on OLTP vs OLAP?


In [17]:
# TODO: result = await Runner.run(your_agent, "...same follow up as Step 6...", session=your_session)
# TODO: print(result.final_output)
result = await Runner.run(study_assistant, "What did I just ask you?", session=study_session)
print(result.final_output)


You said: “I don't know the answer for this one?”


## 8. Give it a real chat UI with Gradio

**TODO:** Copy this skeleton from Lab 7, Step 11 exactly, structure and all. Notice what is
different from your Module 7 assignment's version: `history` is no longer rebuilt inside `chat()`
at all, `session=your_session` already gives `Runner.run` everything it needs to remember the
conversation.

Try a conversation that would trigger more than one of your three tools across a few turns, then
interrupt the cell (stop button) when you are done chatting.

In [18]:
async def chat(message, history):
    # TODO: result = await Runner.run(your_agent, message, session=your_session)
    # TODO: return result.final_output
    pass

gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Reflection Questions

Answer these in a new Markdown cell at the end of your notebook:

1. In Step 6, what exactly did your agent get wrong on the second call, and why? Point to the
   specific piece of information it had no way of knowing.
2. Compare your Gradio `chat()` function here to the one in your Module 7 assignment. What code did
   you get to delete, and what does `session=your_session` do in its place?
3. Your `your_session` was created with a `db_path`. If you had left `db_path` out entirely, what
   would still work the same, and what would break the next time you restarted the kernel?
4. `SQLiteSession` also has a `pop_item()` method, demonstrated in Lab 7, Step 10, for undoing the
   last turn. Describe one realistic situation in your own agent's domain where you would want to
   call it.

## Submission Checklist

- Remove your `.env` file before submitting; it contains your API key.
- Remove any `.db` file your session created; it may contain real conversation content.
- Run the notebook top to bottom and confirm it executes cleanly from a fresh kernel.
- Confirm Step 6 shows the agent failing to remember, and Step 7 shows the same scenario working
  correctly once `session=your_session` is added.
- Answer all four reflection questions in a Markdown cell at the end of the notebook.
- Rename the file to include your name, for example `Module8Assignment_JaneDoe.ipynb`, then submit
  it.

1. In step 6, my agent failed to remember the quiz question from the previous `Runner.run` call because each call started with a blank conversation. The first turn asked me the difference between OLTP and OLAP. The second turn was "Actually, I don't know the answer to that one." Without a session, the agent couldn't tell what my answer was about, and couldn't bookmark the correct topic.

2. I deleted the history that turns each prompt and answer into message and pass to a list. Then I passed `session=study_session` makes `Runner.run` load and append turns from the SQLite session automatically, so `chat()` only needs to send the new `message` and return `result.final_output`.

3. If I left `db_path` out, the memory would still work but it is saved into RAM, when I restart a kernel, that memory will be gone, so the the session memory would start empty. The file `study_session.db` makes the conversation maintain through a restart.

4. One case for `pop_item()` I can think of is a student who accidentally types "I don't know" when they meant to submit an answer. That turn can bookmark that topic even though they may not need to review that topic. By calling `pop_item()` it would undo that turn so they can answer themselves without an extra bookmark in `review_topics.txt`